In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [11]:
data=pd.read_csv('./Churn_Modelling.csv')

In [12]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [13]:
##Preprocess and drop irrelevant features

In [14]:
data.drop(['RowNumber','CustomerId','Surname'],inplace=True,axis=1)

In [15]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [16]:
##Encode the categorical variables
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

In [17]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [18]:
#One hot encode Geography
from sklearn.preprocessing import OneHotEncoder
ohencoder=OneHotEncoder()
geoencoder=ohencoder.fit_transform(data[['Geography']])

In [19]:
geoencoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [20]:
geo_encoded_df=pd.DataFrame(geoencoder.toarray(),columns=ohencoder.get_feature_names_out(['Geography']))

In [21]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [22]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [23]:
## save the encoders and scalers
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehotencoder_geo.pkl','wb') as file:
    pickle.dump(ohencoder,file)

In [24]:
X=data.drop('Exited',axis=1)
y=data['Exited']

In [25]:
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)

In [26]:
scaler=StandardScaler()
Xtr=scaler.fit_transform(Xtr)
Xte=scaler.transform(Xte)

In [27]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [28]:
Xtr

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [29]:
import tensorflow as tf

In [30]:
from tensorflow.keras.models import Sequential

In [31]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [32]:
import datetime

In [33]:
##Build the model

In [34]:
Xtr.shape

(8000, 12)

In [36]:
model=Sequential([
        Dense(64,activation='relu',input_shape=(Xtr.shape[1],)), ## HL 1
        Dense(32,activation='relu'), ## HL 2
        Dense(1,activation='sigmoid') ## Output
])

In [37]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 64)                832       
                                                                 
 dense_3 (Dense)             (None, 32)                2080      
                                                                 
 dense_4 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [38]:
##Compile the model

In [40]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [42]:
model.compile(optimizer=opt,loss='binary_crossentropy',metrics=['accuracy'])

In [43]:
## set up the tensorboard
log_dir='logs/fit'+datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

In [44]:
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [48]:
#Early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [49]:
##TRAIN THE MODEL
history=model.fit(
    Xtr,ytr,validation_data=(Xte,yte),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)



Epoch 1/100


250/250 [==============================] - 3s 5ms/step - loss: 0.3992 - accuracy: 0.8313 - val_loss: 0.3609 - val_accuracy: 0.8545
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3561 - accuracy: 0.8537 - val_loss: 0.3549 - val_accuracy: 0.8560
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3475 - accuracy: 0.8574 - val_loss: 0.3521 - val_accuracy: 0.8590
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3437 - accuracy: 0.8579 - val_loss: 0.3439 - val_accuracy: 0.8620
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3415 - accuracy: 0.8585 - val_loss: 0.3593 - val_accuracy: 0.8555
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 0.3390 - accuracy: 0.8629 - val_loss: 0.3481 - val_accuracy: 0.8585
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 0.3346 - accuracy: 0.8641 - val_loss: 0.3567 - val_accuracy: 0.85

In [50]:
model.save('model.h5')

C:\Users\ASUS\Documents\AI ML Langchain\annclassification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [53]:
##Load tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [55]:
%tensorboard --logdir logs/fit20260228-113807